# 00 — Baseline univariado: pH da estação EF01 (CETESB)

**Objetivo:** estabelecer os baselines que qualquer modelo futuro precisa bater na série de pH (5 min, 01/06–31/08/2026).
**Decisões travadas:** variável `pH` · lookback `L=8640` (30 dias) · horizonte `H=288` (1 dia) · split temporal 70/15/15 pré-holdout sem shuffle · **holdout puro nos últimos 10 dias** (alvos nunca treinados, 10 origens diárias) · baselines clássicos (persistência, sazonal-naive, média móvel, ARIMA em grade horária, Prophet).
**Dados:** `dados/ef01-mogi-das-cruzes_ph_2026-06-01_a_2026-08-31.csv` — ver `dados/README.md` (encoding Windows-1252, `;`, vírgula decimal, ~18% de faltantes, validados até 22/08/2026).

In [1]:
import pickle
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

# --- caminhos (funciona com cwd = repo ou notebooks/) ---
ROOT = Path.cwd() if (Path.cwd() / "dados").exists() else Path.cwd().parent
CSV = ROOT / "dados" / "ef01-mogi-das-cruzes_ph_2026-06-01_a_2026-08-31.csv"
OUT = ROOT / "resultados" / "00-baseline-ph"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# --- hiperparâmetros do experimento ---
L, H = 8640, 288          # lookback (30 dias) e horizonte (1 dia) em passos de 5 min
SEASON = 288              # ciclo diário em passos de 5 min
INTERP_LIMIT = 24         # interpolação temporal máx. (24 passos = 2 h)
PROVISORIO_CORTE = "2026-08-22 09:00"  # dados após isso não são validados
HOLDOUT_DIAS = 10          # cauda final separada como holdout puro (só alvos)
ARIMA_ORDER = (2, 1, 2)   # ARIMA roda em grade horária (L=720h, H=24h): 5 min × L=8640 é inviável
ARIMA_STRIDE = 48         # ARIMA reestimado a cada 48 origens do teste rolante

print("ROOT:", ROOT, "| CSV existe:", CSV.exists())

ROOT: /home/marcos/Projetos/temporal-model | CSV existe: True


## 1. Carga
Formato CETESB: `;`, decimal com vírgula, `windows-1252`, linha 1 = validação, linha 2 = cabeçalho.

In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
df = df.rename(columns={"Data hora": "ds", "pH": "y"}).sort_values("ds").reset_index(drop=True)
print(df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()

(26209, 2) 2026-06-01 00:00:00 → 2026-08-31 00:00:00
faltantes: 4794 (18.3%)


,ds,y
count,26209,21415.000000
mean,2026-07-16 12:00:00,6.142466
min,2026-06-01 00:00:00,5.560000
25%,2026-06-23 18:00:00,5.970000
50%,2026-07-16 12:00:00,6.170000
75%,2026-08-08 06:00:00,6.280000
max,2026-08-31 00:00:00,6.590000
std,NaN,0.198893


## 2. EDA — perfil, faltantes e ciclo diário

In [3]:
isna = df["y"].isna().to_numpy()
gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
print(f"maior gap: {gaps.max()} passos = {gaps.max()*5/60:.1f} h | gaps > 24 passos: {(gaps > 24).sum()}")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.4)
ax[0].axvline(pd.Timestamp(PROVISORIO_CORTE), color="r", ls="--", lw=1)
ax[0].set_title("pH EF01 — série completa (vermelho = início do trecho provisório)")
ax[0].set_ylabel("pH")
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição do pH")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("pH por hora do dia (ciclo diário?)")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva:", OUT / "figs" / "01-eda.png")

maior gap: 18 passos = 1.5 h | gaps > 24 passos: 0


fig salva: /home/marcos/Projetos/temporal-model/resultados/00-baseline-ph/figs/01-eda.png


## 3. Limpeza — grade completa + interpolação limitada
Reindex na grade de 5 min, flag do trecho provisório e interpolação temporal de no máximo 2 h. Gaps maiores permanecem `NaN` e as janelas que os contêm são descartadas (sem vazamento).

In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_raw = df.set_index("ds")["y"].reindex(idx)
print(f"slots na grade: {len(s_raw)} | linhas no CSV: {len(df)} (iguais = nenhum timestamp ausente)")
s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")
provisorio = s.index > pd.Timestamp(PROVISORIO_CORTE)
print(f"trecho provisório: {int(provisorio.sum())} slots ({100*provisorio.mean():.1f}%)")

amostra = slice("2026-06-08", "2026-06-15")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_raw[amostra].index, s_raw[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 08–15/06")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")

slots na grade: 26209 | linhas no CSV: 26209 (iguais = nenhum timestamp ausente)
NaN após interpolação (limite 24): 0
trecho provisório: 2484 slots (9.5%)


fig salva


## 4. Estacionariedade (ADF) e decomposição STL
STL roda nos últimos 4032 pontos do treino (rápido e representativo).

In [5]:
n_total = len(s)
n_train = int(n_total * 0.70)
train = s.iloc[:n_train].dropna()
stat, pval, *_ = adfuller(train.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária (usar diferenciação / modelos robustos)'}")

stl = STL(train.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")

ADF stat=-4.66 p-valor=0.000102 → estacionária


fig salva


## 5. Janelamento + holdout puro
Amostras `(L=8640 → H=288)` por janela deslizante, só janelas 100% observadas. Pré-holdout: split 70/15/15 **sem shuffle**. Holdout: últimos 10 dias — **alvos nunca treinados** (o contexto de 30 dias pode alcançar o treino, o alvo não). Mais 10 origens diárias (fim de cada dia previsto) para o teste dia-a-dia.

In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]             # timestamp do último alvo de cada janela
n = len(X)
ZONE = s.index.max() - pd.Timedelta(days=HOLDOUT_DIAS)
is_hold = ends >= (ZONE + pd.Timedelta(minutes=5 * (H - 1)))
ho = np.where(is_hold)[0]
pre = np.where(~is_hold)[0]
i1, i2 = int(len(pre) * 0.70), int(len(pre) * 0.85)
tr, va, te = pre[:i1], pre[i1:i2], pre[i2:]
splits = {"train": tr, "val": va, "test": te, "holdout": ho}
for k, idx in splits.items():
    print(f"{k}: {len(idx)} janelas | alvos {ends[idx[0]].date()} → {ends[idx[-1]].date()}")
print(f"janelas descartadas (com NaN): {len(s) - L - H + 1 - n}")
print(f"zona holdout (alvos): {ZONE.date()} → {s.index.max().date()}")
# 10 origens diárias: fim de cada dia previsto no holdout
daily_ends = [ZONE + pd.Timedelta(minutes=5 * (H - 1 + H * k)) for k in range(HOLDOUT_DIAS)]
daily_idx = np.array([int(np.where(ends == d)[0][0]) for d in daily_ends])
print("dias previstos:", [str(ends[i].date()) for i in daily_idx])
TR_END = ends[tr[-1]]   # corte do treino p/ modelos globais (Prophet)

train: 10281 janelas | alvos 2026-07-01 → 2026-08-06
val: 2203 janelas | alvos 2026-08-06 → 2026-08-14
test: 2204 janelas | alvos 2026-08-14 → 2026-08-21
holdout: 2594 janelas | alvos 2026-08-21 → 2026-08-31
janelas descartadas (com NaN): 0
zona holdout (alvos): 2026-08-21 → 2026-08-31
dias previstos: ['2026-08-21', '2026-08-22', '2026-08-23', '2026-08-24', '2026-08-25', '2026-08-26', '2026-08-27', '2026-08-28', '2026-08-29', '2026-08-30']


## 6. Baselines baratos (teste rolante + holdout)
Persistência, sazonal-naive (lag 288 = ontem-na-mesma-hora) e média móvel das últimas 288 obs. Vetorizados: rodam em todas as origens.

In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

Xte, Yte = X[te], Y[te]
Xho, Yho = X[ho], Y[ho]
pred_te = cheap_preds(Xte)
pred_ho = cheap_preds(Xho)
print("teste rolante:")
print(pd.DataFrame({m: metricas(Yte, p) for m, p in pred_te.items()}).T.round(4).to_string())
print("holdout (todas as origens):")
print(pd.DataFrame({m: metricas(Yho, p) for m, p in pred_ho.items()}).T.round(4).to_string())

teste rolante:
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0727  0.0942  1.1683  1.1657
sazonal_naive_288  0.0501  0.0648  0.8079  0.8051
media_movel_288    0.0631  0.0768  1.0159  1.0133
holdout (todas as origens):


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0875  0.1135  1.4073  1.4061
sazonal_naive_288  0.0460  0.0661  0.7395  0.7394
media_movel_288    0.0669  0.0837  1.0748  1.0745


## 7. ARIMA em grade horária (subamostras — custo)
ARIMA(2,1,2) sobre a série reamostrada para 1 h (`L=720h`, `H=24h`); cada previsão horária é repetida 12× para voltar aos 5 min (aproximação documentada). Subamostra do teste rolante (stride) + as 10 origens diárias do holdout.

In [8]:
hs = s.resample("1h").mean()

def arima_hora(e):
    he = e.floor("h")
    ctx = hs.loc[he - pd.Timedelta(hours=719):he].values
    fc = ARIMA(ctx, order=ARIMA_ORDER).fit().get_forecast(24).predicted_mean.values
    return np.repeat(fc, 12)[:H]

def roda_arima(idxs, nome):
    P = np.empty((len(idxs), H))
    t0 = time.time()
    for j, i in enumerate(idxs):
        try:
            P[j] = arima_hora(ends[i])
        except Exception:
            P[j] = np.repeat(X[i, -1], H)   # fallback honesto: persistência
        if (j + 1) % 10 == 0:
            print(f"  {nome}: {j+1}/{len(idxs)} origens...", flush=True)
    print(f"ARIMA {nome}: {len(idxs)} origens em {time.time()-t0:.0f}s")
    return P

idx_a = np.arange(0, len(te), ARIMA_STRIDE)
Pa = roda_arima(te[idx_a], "teste")
print(metricas(Y[te[idx_a]], Pa))
Pd = roda_arima(daily_idx, "holdout-diario")
print("holdout diário:", metricas(Y[daily_idx], Pd))

# artefato: ARIMA ajustado na cauda horária do treino (inspeção/reuso)
h_tr = hs.loc[:TR_END].iloc[-720:].values
with open(OUT / "modelos" / "arima212_cauda_treino.pkl", "wb") as f:
    pickle.dump(ARIMA(h_tr, order=ARIMA_ORDER).fit(), f)
print("modelo salvo")

  teste: 10/46 origens...


  teste: 20/46 origens...


  teste: 30/46 origens...


  teste: 40/46 origens...


ARIMA teste: 46 origens em 73s
{'MAE': 0.0724322161835749, 'RMSE': 0.09369623659686371, 'MAPE': 1.1637794127855439, 'sMAPE': 1.16255209860523}


  holdout-diario: 10/10 origens...


ARIMA holdout-diario: 10 origens em 17s
holdout diário: {'MAE': 0.09733333333333336, 'RMSE': 0.12036056940155558, 'MAPE': 1.5773945700406886, 'sMAPE': 1.5598883107281725}


modelo salvo


## 8. Prophet (opcional — pula se `prophet`/CmdStan indisponível)
Um único ajuste no treino (tolera `NaN`) e projeção sobre teste + holdout; métricas nas mesmas janelas.

In [9]:
PROPHET_OK = False
try:
    from prophet import Prophet
    import cmdstanpy
    assert cmdstanpy.cmdstan_path() is not None
    df_train = pd.DataFrame({"ds": s.loc[:TR_END].index, "y": s.loc[:TR_END].values}).dropna()
    m = Prophet(daily_seasonality=True, weekly_seasonality=True)
    m.fit(df_train)
    fmap = m.predict(pd.DataFrame({"ds": s.index})).set_index("ds")["yhat"]

    def fatia(idxs):
        E = ends[idxs]
        return np.stack([[fmap.loc[d - pd.Timedelta(minutes=5*(H-1-h))] for h in range(H)] for d in E])

    Pp_te, Pp_ho, Pp_d = fatia(te), fatia(ho), fatia(daily_idx)
    PROPHET_OK = True
    print("Prophet teste:", metricas(Yte, Pp_te))
    print("Prophet holdout:", metricas(Yho, Pp_ho))
    from prophet.serialize import model_to_json
    (OUT / "modelos" / "prophet_ph.json").write_text(model_to_json(m))
    print("modelo salvo")
except Exception as e:
    print(f"Prophet pulado ({type(e).__name__}: {str(e)[:150]}).")

Importing plotly failed. Interactive plots will not work.


19:20:38 - cmdstanpy - INFO - Chain [1] start processing


19:20:56 - cmdstanpy - INFO - Chain [1] done processing


Prophet teste: {'MAE': 0.05819841085029332, 'RMSE': 0.07120890262976796, 'MAPE': 0.9347410182325535, 'sMAPE': 0.9336738811955854}
Prophet holdout: {'MAE': 0.0492614509552443, 'RMSE': 0.06502334136400403, 'MAPE': 0.7885016922184823, 'sMAPE': 0.7917255289583603}
modelo salvo


## 9. Comparação final + holdout dia a dia
Tabela do teste rolante, tabela do holdout diário (todos os modelos, mesmos alvos) e MAE por dia previsto. A régua do projeto é impressa abaixo.

In [10]:
linhas = {m: metricas(Yte, p) for m, p in pred_te.items()}
if PROPHET_OK:
    linhas["prophet"] = metricas(Yte, Pp_te)
tab = pd.DataFrame(linhas).T.round(4)
tab.to_csv(OUT / "metricas_baseline.csv")
print("=== teste rolante ===")
print(tab.to_string())

# holdout diário: todos os modelos nos mesmos 10 dias
Yd = Y[daily_idx]
diario = {m: metricas(Yd, cheap_preds(X[daily_idx])[m]) for m in pred_te}
diario["arima_212_h"] = metricas(Yd, Pd)
if PROPHET_OK:
    diario["prophet"] = metricas(Yd, Pp_d)
tab_d = pd.DataFrame(diario).T.round(4)
tab_d.to_csv(OUT / "metricas_holdout.csv")
print("=== holdout diário (10 dias) ===")
print(tab_d.to_string())

# MAE por dia previsto
por_dia = pd.DataFrame(
    {m: [mae(Yd[k:k+1], cheap_preds(X[daily_idx])[m][k:k+1]) for k in range(len(Yd))]
     for m in pred_te},
    index=[str(ends[i].date()) for i in daily_idx])
por_dia["arima_212_h"] = [mae(Yd[k:k+1], Pd[k:k+1]) for k in range(len(Yd))]
if PROPHET_OK:
    por_dia["prophet"] = [mae(Yd[k:k+1], Pp_d[k:k+1]) for k in range(len(Yd))]
print(por_dia.round(4).to_string())
print(f"\nRégua (menor MAE no teste rolante): {tab['MAE'].idxmin()} = {tab['MAE'].min():.4f}")
print(f"Régua (menor MAE no holdout diário): {tab_d['MAE'].idxmin()} = {tab_d['MAE'].min():.4f}")

=== teste rolante ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0727  0.0942  1.1683  1.1657
sazonal_naive_288  0.0501  0.0648  0.8079  0.8051
media_movel_288    0.0631  0.0768  1.0159  1.0133
prophet            0.0582  0.0712  0.9347  0.9337
=== holdout diário (10 dias) ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0973  0.1204  1.5774  1.5599
sazonal_naive_288  0.0466  0.0673  0.7501  0.7497
media_movel_288    0.0655  0.0823  1.0547  1.0540
arima_212_h        0.0973  0.1204  1.5774  1.5599
prophet            0.0474  0.0632  0.7599  0.7626
            persistencia  sazonal_naive_288  media_movel_288  arima_212_h  prophet
2026-08-21        0.0642             0.0317           0.0562       0.0642   0.0307
2026-08-22        0.0686             0.0363           0.0485       0.0686   0.0364
2026-08-23        0.0725             0.0444           0.0631       0.0725   0.0426
2026-08-24        0.0972             0.0414           0.0508   

In [11]:
E = ends[te]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
for ax, k in zip(axes, [0, len(Xte)//2, -1]):
    tc = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H+2015)), E[k] - pd.Timedelta(minutes=5*H), freq="5min")
    ax.plot(tc, Xte[k][-2016:], lw=0.8, label="contexto (cauda 7d)")
    tf = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H-1)), E[k], freq="5min")
    ax.plot(tf, Yte[k], "k-", lw=1.5, label="real")
    ax.plot(tf, pred_te["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, pred_te["persistencia"][k], ":", lw=1, label="persistência")
    ax.set_title(f"origem {E[k]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

fig, ax = plt.subplots(figsize=(8, 4))
tab["MAE"].sort_values().plot.barh(ax=ax)
ax.set_title("MAE no teste rolante — baselines (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

# 10 dias previstos × real (o teste que o usuário pediu)
fig, axes = plt.subplots(5, 2, figsize=(14, 12), sharey=False)
for ax, k in zip(axes.ravel(), range(len(Yd))):
    tf = pd.date_range(ends[daily_idx[k]] - pd.Timedelta(minutes=5*(H-1)), ends[daily_idx[k]], freq="5min")
    ax.plot(tf, Yd[k], "k-", lw=1.2, label="real")
    ax.plot(tf, cheap_preds(X[daily_idx])["persistencia"][k], ":", lw=1, label="persistência")
    ax.plot(tf, cheap_preds(X[daily_idx])["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    if PROPHET_OK:
        ax.plot(tf, Pp_d[k], "-.", lw=1, label="prophet")
    ax.plot(tf, Pd[k], lw=1, alpha=0.7, label="arima-h")
    ax.set_title(f"dia previsto {ends[daily_idx[k]].date()} (MAE pers={por_dia['persistencia'].iloc[k]:.3f})")
    ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-holdout-dias.png")
print("figs salvas")

figs salvas


## 10. Conclusões e próximos passos

- A **régua** (menor MAE, impressa na §9) vale para teste rolante e holdout separadamente: LSTM/GRU (§7 do README) e depois TFT/PatchTST (§3) precisam superá-la **neste desenho**.
- Em H=1 dia a persistência deve perder para o sazonal-naive — se não perder, o desenho (ou os dados) será revisto antes de fechar.
- ARIMA roda em grade horária com expansão ×12 (aproximação documentada na §7); compare-o só na tabela do subconjunto/diária.
- Artefatos em `resultados/00-baseline-ph/`: `metricas_baseline.csv`, `metricas_holdout.csv`, `modelos/` (ARIMA + Prophet) e `figs/` (inclui `06-holdout-dias.png`).